# KERAS RNN trên retailrocket

Nguyễn Ngọc Hoàng Nam — B23DCCN585 | ASG 06

Các cell nhỏ được sắp theo thứ tự phụ thuộc. Huấn luyện đầy đủ mặc định tắt; số liệu chạy thử không phải kết quả test chính thức.

In [1]:
from pathlib import Path
import sys, json, os
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
assert (ROOT/"src").is_dir()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.paths import DATA
from src.io import read_json
import numpy as np
import pandas as pd
from IPython.display import display, Image
print("Project:", ROOT)
print("Data:", DATA)


Project: E:\PTHTTM\ASG_06
Data: E:\PTHTTM\ASG_06_data


## 1. Dữ liệu và hình dạng

In [2]:
from src.data import load_data,split_ids,make_batch
data=load_data('retailrocket')
x,lengths,y=make_batch(data,split_ids(data,'train')[:8])
print(x.shape,lengths,y)

(8, 20, 10) [1 2 3 4 5 6 7 1] [0 0 0 0 0 0 0 0]


## 2. Hàm xây dựng RNN cơ bản

In [3]:
from src.backends import keras_runtime
tf=keras_runtime(42,cpu=True)
print(tf.__version__)

2.10.1


In [4]:
def build_keras(tf, input_dim, hidden, output_dim, length):
    x=tf.keras.Input((length,input_dim),name="sequence")
    sizes=tf.keras.Input((),dtype="int32",name="length")
    mask=tf.sequence_mask(sizes,maxlen=length)
    h=tf.keras.layers.SimpleRNN(hidden,activation="tanh",name="rnn")(x,mask=mask)
    output=tf.keras.layers.Dense(output_dim,name="head")(h)
    return tf.keras.Model([x,sizes],output)

In [5]:
model=build_keras(tf,data['metadata']['input_dim'],32,3,data['metadata']['sequence_length'])
model.summary()

Model: "model"


__________________________________________________________________________________________________


 Layer (type)                   Output Shape         Param #     Connected to                     


 length (InputLayer)            [(None,)]            0           []                               


 sequence (InputLayer)          [(None, 20, 10)]     0           []                               


...

 head (Dense)                   (None, 3)            99          ['rnn[0][0]']                    


Total params: 1,475


Trainable params: 1,475


Non-trainable params: 0


__________________________________________________________________________________________________


## 3. Lan truyền thuận và hàm mất mát

In [6]:
output=model([x,lengths],training=False)
print('Output:',output.shape)

Output: (8, 3)


In [7]:
with tf.GradientTape() as tape:
    output=model([x,lengths],training=True)
    loss=tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(labels=tf.one_hot(y,3),logits=output))
gradients=tape.gradient(loss,model.trainable_variables)
print('Loss minh họa:',float(loss))
print('Norm gradient:',float(tf.linalg.global_norm(gradients)))

Loss minh họa: 1.1490068435668945
Norm gradient: 2.987266778945923


## 4. Bộ huấn luyện đầy đủ
Ví dụ trên giải thích đồ thị. Bộ huấn luyện dùng cùng họ kiến trúc, khởi tạo chung giữa framework, trọng số lớp cho Retailrocket, clipping và checkpoint. Mã thuật toán dùng cho lượt chính nằm trong src/backends.py và src/training.py.

In [8]:
from src.training import configuration
config,_=configuration('retailrocket','keras')
display(config)

{'seed': 42,
 'hidden_size': 32,
 'batch_size': 256,
 'max_epochs': 20,
 'patience': 4,
 'min_delta': 1e-05,
 'learning_rate': 0.001,
 'adam_epsilon': 1e-07,
 'clip_norm': 1,
 'dataset': 'retailrocket',
 'framework': 'keras',
 'smoke': False,
 'class_weights': [0.778438758214022, 3.0554776934025285, 5.659938637800053],
 'metadata_sha256': '08e6797558192e232e85cf83c11f7dea1b86ea1593586222fbcc27a4b0845620',
 'selection': 'validation macro-F1'}

## 5. Chủ động chạy cấu hình này
Chỉ chuyển RUN_FULL=True khi muốn huấn luyện đầy đủ. Nên dùng RUN_TRAINING.bat để giảm bộ nhớ của kernel minh họa đang mở.

In [9]:
RUN_FULL=False
if RUN_FULL:
    import subprocess
    environment=os.environ.copy()
    environment.pop('CUDA_VISIBLE_DEVICES',None)
    subprocess.run([sys.executable,'-m','src.training','--dataset','retailrocket','--framework','keras'],cwd=ROOT,env=environment,check=True)
else:
    print('Chưa khởi động huấn luyện đầy đủ.')

Chưa khởi động huấn luyện đầy đủ.


## 6. Dự đoán và kết quả được lưu

In [10]:
folder=ROOT/'results/runs/retailrocket_keras_seed42'
if (folder/'metrics.json').exists():
    display(read_json(folder/'metrics.json'))
    display(pd.read_csv(folder/'predictions.csv').head(12))
else:
    print('Kết quả chính sẽ có sau khi bạn huấn luyện xong.')

Kết quả chính sẽ có sau khi bạn huấn luyện xong.
